# Clinical Data Preprocessing

TCGA-BRCA clinical data preprocessing pipeline.

**Output files** (saved to `Clinical/`):
- `clinical_filtered_raw.csv` — filtered, indexed by patient ID
- `clinical_selected.csv` — after manual variable selection
- `clinical_preprocessed.csv` — encoded (no prefix)
- `clinical_preprocessed_prefixed.csv` — encoded with `CLIN_` prefix, **use this for merge**

In [2]:
"""
Clinical data preprocessing pipeline for TCGA-BRCA.
Input:  data/TCGA-BRCA.clinical.tsv
Output: Clinical/clinical_filtered_raw.csv
        Clinical/clinical_selected.csv
        Clinical/clinical_preprocessed.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path

# BASE_DIR is the project root, resolved relative to this script.
# Expected layout (regardless of where the project is saved):
#   <project_root>/
#       data/TCGA-BRCA.clinical.tsv
#       RNA/preprocessed/outcome.csv
#       Clinical/                        <- output (created automatically)
#       01_Causal_feature_extraction/
#           Clinical/
#               clinical_preprocessing.py  <- this script
#
# To use: place this script inside the project and run it — no path changes needed.

try:
    # Running as a .py script: path relative to script location
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    # Running in Jupyter: assumes notebook is in 01_Causal_feature_extraction/Clinical/
    BASE_DIR = Path.cwd().parents[1]
CLIN_FILE    = BASE_DIR / "data" / "TCGA-BRCA.clinical.tsv"
OUT_DIR      = BASE_DIR / "01_Causal_feature_extraction" / "Clinical"
OUT_DIR.mkdir(exist_ok=True)

print(f"Project root:  {BASE_DIR}")
print(f"Clinical file: {CLIN_FILE.exists()} -> {CLIN_FILE}")


# ============================================================================
# STEP 1: LOAD
# ============================================================================

clinical = pd.read_csv(CLIN_FILE, sep="\t", low_memory=False)

print(f"Clinical raw:  {clinical.shape}")


# ============================================================================
# STEP 2: KEEP TUMOR SAMPLES ONLY, THEN DEDUPLICATE
# ============================================================================

clinical = clinical[clinical["tissue_type.samples"] == "Tumor"].copy()
print(f"After Tumor filter: {clinical.shape}")

clinical = clinical.drop_duplicates(subset="submitter_id", keep="first")
print(f"After dedup:        {clinical.shape}")


# ============================================================================
# STEP 3: REMOVE COLUMNS WITH >= 80% MISSING
# ============================================================================

MISSING_THRESHOLD = 80
pct_missing = clinical.isnull().sum() / len(clinical) * 100
cols_keep   = pct_missing[pct_missing < MISSING_THRESHOLD].index.tolist()
clinical    = clinical[cols_keep].copy()
print(f"After missing filter: {clinical.shape}")


# ============================================================================
# STEP 4: REMOVE METADATA / ADMIN COLUMNS
# ============================================================================

METADATA_KEYWORDS = [
    "uuid", "barcode", "project_id", "program.", "tissue_source_site",
    "created_datetime", "updated_datetime", "state",
    "sample_type.samples", "sample_id.samples",
]

metadata_cols = [
    col for col in clinical.columns
    if any(kw in col.lower() for kw in METADATA_KEYWORDS)
    and col != "submitter_id"
    and "treatment"   not in col.lower()
    and "diagnos"     not in col.lower()
    and "demographic" not in col.lower()
]
clinical = clinical.drop(columns=metadata_cols)
print(f"After metadata drop:  {clinical.shape}")


# ============================================================================
# STEP 5: REMOVE CONSTANT COLUMNS
# ============================================================================

constant_cols = [
    col for col in clinical.columns
    if col != "submitter_id"
    and clinical[col].nunique(dropna=True) == 1
]
clinical = clinical.drop(columns=constant_cols)
print(f"After constant drop:  {clinical.shape}")


# ============================================================================
# STEP 6: SET INDEX = submitter_id
# Patient alignment to outcome happens at merge stage.
# ============================================================================

clinical = clinical.set_index("submitter_id")
print(f"Samples after all filters: {clinical.shape}")

clinical.to_csv(OUT_DIR / "clinical_filtered_raw.csv")
print("Saved: clinical_filtered_raw.csv")


# ============================================================================
# STEP 7: MANUAL VARIABLE SELECTION
#   - drop redundant age / date columns
#   - drop low-information sample processing columns
#   - drop leakage columns (vital_status, days_to_last_follow_up encode the outcome)
# ============================================================================

DROP_COLS = [
    # Unique IDs / admin
    "id",
    "sample",
    "case_id",
    "treatment_id.treatments.diagnoses",
    "submitter_id.treatments.diagnoses",
    "created_datetime.treatments.diagnoses",   # 1072 unique - not a feature
    "updated_datetime.treatments.diagnoses",   # 1072 unique - not a feature

    # Redundant age representations (keep age_at_index only)
    "age_at_diagnosis.diagnoses",
    "age_at_earliest_diagnosis.diagnoses.xena_derived",
    "age_at_earliest_diagnosis_in_years.diagnoses.xena_derived",
    "days_to_birth.demographic",
    "year_of_birth.demographic",

    # Non-predictive / redundant
    "year_of_diagnosis.diagnoses",
    "days_to_collection.samples",
    "icd_10_code.diagnoses",
    "site_of_resection_or_biopsy.diagnoses",

    # Sample processing metadata
    "is_ffpe.samples",
    "oct_embedded.samples",
    "preservation_method.samples",
    "specimen_type.samples",
    "sample_type_id.samples",
    "initial_weight.samples",

    # Low information
    "synchronous_malignancy.diagnoses",
    "treatment_type.treatments.diagnoses",

    # LEAKAGE: directly encode the outcome (OS / OS.time)
    "vital_status.demographic",
    "days_to_last_follow_up.diagnoses",
]

DROP_COLS = [c for c in DROP_COLS if c in clinical.columns]
clinical  = clinical.drop(columns=DROP_COLS)
print(f"After manual selection: {clinical.shape}")

clinical.to_csv(OUT_DIR / "clinical_selected.csv")
print("Saved: clinical_selected.csv")


# ============================================================================
# STEP 8: ENCODING
# ============================================================================

NUMERIC_VARS     = ["age_at_index.demographic"]
categorical_vars = [c for c in clinical.columns if c not in NUMERIC_VARS]
binary_vars      = [c for c in categorical_vars if clinical[c].nunique() == 2]
multi_vars       = [c for c in categorical_vars if clinical[c].nunique() > 2]

print(f"\nNumeric:       {len(NUMERIC_VARS)}")
print(f"Binary (0/1):  {len(binary_vars)}")
print(f"Multi (OHE):   {len(multi_vars)}")

# --- Standardize missing values ---
MISSING_MAP = {"--": "missing", "not reported": "missing",
               "Not Reported": "missing", "unknown": "missing",
               "Unknown": "missing"}

clinical_enc = clinical.copy()

for col in categorical_vars:
    clinical_enc[col] = clinical_enc[col].fillna("missing").replace(MISSING_MAP)

# --- Impute numeric ---
for col in NUMERIC_VARS:
    if clinical_enc[col].isnull().any():
        median = clinical_enc[col].median()
        clinical_enc[col] = clinical_enc[col].fillna(median)
        print(f"Imputed {col} with median={median:.1f}")

# --- Encode binary variables (0/1) ---
for col in binary_vars:
    vals = clinical_enc[col].unique()
    if "Alive" in vals and "Dead" in vals:
        mapping = {"Alive": 0, "Dead": 1}
    elif set(str(v).lower() for v in vals) <= {"male", "female"}:
        mapping = {v: (0 if str(v).lower() == "female" else 1) for v in vals}
    elif "Tumor" in vals:
        mapping = {v: (1 if v == "Tumor" else 0) for v in vals}
    else:
        sorted_vals = sorted(str(v) for v in vals)
        mapping = {sorted_vals[0]: 0, sorted_vals[1]: 1}
    clinical_enc[col] = clinical_enc[col].map(mapping)
    print(f"Binary: {col}  ->  {mapping}")

# --- One-hot encode multi-category variables ---
print("\nUnique values per multi-category column (before OHE):")
for col in multi_vars:
    print(f"  {clinical_enc[col].nunique():4d}  {col}")

clinical_enc = pd.get_dummies(clinical_enc, columns=multi_vars, dtype=int)

# --- Verify ---
n_missing = clinical_enc.isnull().sum().sum()
print(f"\nMissing values after encoding: {n_missing}")
print(f"Final shape: {clinical_enc.shape}")

clinical_enc.to_csv(OUT_DIR / "clinical_preprocessed.csv")
print("Saved: clinical_preprocessed.csv")


# ============================================================================
# SUMMARY
# ============================================================================

# Add CLIN_ prefix and save prefixed version (used for merge)
clinical_prefixed = clinical_enc.copy()
clinical_prefixed.columns = [f"CLIN_{c}" for c in clinical_prefixed.columns]
clinical_prefixed.to_csv(OUT_DIR / "clinical_preprocessed_prefixed.csv")
print("Saved: clinical_preprocessed_prefixed.csv")

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"Samples:  {clinical_enc.shape[0]}")
print(f"Features: {clinical_enc.shape[1]}")
print(f"\n  Numeric:        {len(NUMERIC_VARS)}")
print(f"  Binary:         {len(binary_vars)}")
print(f"  OHE expanded:   {clinical_enc.shape[1] - len(NUMERIC_VARS) - len(binary_vars)}")
print(f"\nOutput files in: {OUT_DIR}")
print(f"  clinical_filtered_raw.csv")
print(f"  clinical_selected.csv")
print(f"  clinical_preprocessed.csv        (no prefix)")
print(f"  clinical_preprocessed_prefixed.csv (CLIN_ prefix, use for merge)")

Project root:  C:\Users\olegk\Desktop\Thesis_v3
Clinical file: True -> C:\Users\olegk\Desktop\Thesis_v3\data\TCGA-BRCA.clinical.tsv
Clinical raw:  (1255, 85)
After Tumor filter: (1118, 85)
After dedup:        (1098, 85)
After missing filter: (1098, 67)
After metadata drop:  (1098, 57)
After constant drop:  (1098, 44)
Samples after all filters: (1098, 43)
Saved: clinical_filtered_raw.csv
After manual selection: (1098, 17)
Saved: clinical_selected.csv

Numeric:       1
Binary (0/1):  2
Multi (OHE):   14
Imputed age_at_index.demographic with median=58.0
Binary: gender.demographic  ->  {'female': 0, 'male': 1}
Binary: tumor_descriptor.samples  ->  {'Metastatic': 0, 'Primary': 1}

Unique values per multi-category column (before OHE):
     9  disease_type
     5  race.demographic
     3  ethnicity.demographic
    13  ajcc_pathologic_stage.diagnoses
     7  tissue_or_organ_of_origin.diagnoses
    23  primary_diagnosis.diagnoses
     3  prior_malignancy.diagnoses
     3  prior_treatment.diagno